# Fire HLS NDVI Timelapse

This notebook creates a timelapse of NDVI from HLS Sentinel-2 data for a specific forest fire.

Steps:
1. Load fire contours shapefile and select fire by CLE ID.
2. Compute expanded bounding box (2x width and height).
3. Define time range (±2 months from fire start date).
4. Search and download HLS Sentinel-2 VI products using earthaccess.
5. Extract NDVI and create timelapse animation.
6. Save animation to data folder.


In [ ]:
# Import necessary libraries
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import box
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import re
import glob
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import colors
import earthaccess
import rioxarray as rxr
import xarray as xr
import warnings
warnings.filterwarnings('ignore')

# USER-SPECIFIED THRESHOLDS (modify these values as needed)
FIRE_COVERAGE_THRESHOLD = 80.0  # Minimum % of fire boundary that must be covered
VALID_PIXEL_THRESHOLD = 80.0    # Minimum % of valid pixels required in image

# Set fire ID (change this to the desired fire CLE ID)
FIRE_CLE_ID = '20211080450'  # Replace with actual CLE value

# Paths
SHAPEFILE_PATH = '../data/shapefiles/fire_contours/FEUX_CONT_SIMP_1972_2022.shp'
OUTPUT_DIR = f'../data/HLS_data/{FIRE_CLE_ID}'
ANIMATION_OUTPUT = f'../data/fire_timelapse_{FIRE_CLE_ID}.gif'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load fire contours using pyogrio engine (to avoid fiona compatibility issues)
print("Loading fire contours using pyogrio engine...")

try:
    fire_gdf = gpd.read_file(SHAPEFILE_PATH, engine='pyogrio')
    print(f"Successfully loaded {len(fire_gdf)} fire records using pyogrio.")
except Exception as e:
    print(f"Error loading shapefile with pyogrio: {e}")
    print("\nTo fix this issue, you can:")
    print("1. Install pyogrio: pip install pyogrio")
    print("2. Ensure you have GDAL installed properly")
    raise

# Check if CLE column exists
if 'CLE' not in fire_gdf.columns:
    # Try alternative column names
    possible_cle_cols = [col for col in fire_gdf.columns if 'cle' in col.lower()]
    if possible_cle_cols:
        CLE_COL = possible_cle_cols[0]
        print(f"Using column '{CLE_COL}' for CLE ID.")
    else:
        raise ValueError("CLE column not found in shapefile. Available columns: " + str(list(fire_gdf.columns)))
else:
    CLE_COL = 'CLE'

# Select fire by CLE ID
fire_row = fire_gdf[fire_gdf[CLE_COL] == FIRE_CLE_ID]
if fire_row.empty:
    raise ValueError(f"Fire CLE {FIRE_CLE_ID} not found.")
else:
    print(f"Found fire in main shapefile.")

print(f"Selected fire: {fire_row.iloc[0].to_dict()}")

In [ ]:
# Get geometry and compute expanded bounding box in EPSG:4326 for earthaccess
geom = fire_row.geometry.iloc[0]

# Get original bounds in current CRS
minx_orig, miny_orig, maxx_orig, maxy_orig = geom.bounds
print(f"Original bounds in {fire_gdf.crs}: ({minx_orig}, {miny_orig}) to ({maxx_orig}, {maxy_orig})")

# Reproject to EPSG:4326 (WGS84) for earthaccess if needed
if fire_gdf.crs != "EPSG:4326":
    print(f"Reprojecting geometry from {fire_gdf.crs} to EPSG:4326 for earthaccess compatibility")
    # Create a temporary GeoSeries with the geometry to reproject it
    from geopandas import GeoSeries
    temp_gs = GeoSeries([geom], crs=fire_gdf.crs)
    geom_wgs84 = temp_gs.to_crs("EPSG:4326").iloc[0]
    minx, miny, maxx, maxy = geom_wgs84.bounds
else:
    minx, miny, maxx, maxy = minx_orig, miny_orig, maxx_orig, maxy_orig

print(f"Bounds in EPSG:4326: ({minx}, {miny}) to ({maxx}, {maxy})")

# Calculate center and dimensions
center_x = (minx + maxx) / 2
center_y = (miny + maxy) / 2
width = maxx - minx
height = maxy - miny

# Expand to twice the width and height
new_width = width * 2
new_height = height * 2
new_minx = center_x - new_width / 2
new_maxx = center_x + new_width / 2
new_miny = center_y - new_height / 2
new_maxy = center_y + new_height / 2

print(f"Expanded bounds in EPSG:4326: ({new_minx}, {new_miny}) to ({new_maxx}, {new_maxy})")

# Store bounding box for earthaccess (format: (west, south, east, north))
bbox = (new_minx, new_miny, new_maxx, new_maxy)

In [ ]:
# Get fire start date and define time range
if 'DATE_DEBUT' in fire_row.columns:
    fire_start = pd.to_datetime(fire_row['DATE_DEBUT'].iloc[0])
elif 'date_debut' in fire_row.columns:
    fire_start = pd.to_datetime(fire_row['date_debut'].iloc[0])
else:
    # Try to find any date column
    date_cols = [col for col in fire_row.columns if 'date' in col.lower() and ('debut' in col.lower() or 'start' in col.lower())]
    if date_cols:
        fire_start = pd.to_datetime(fire_row[date_cols[0]].iloc[0])
    else:
        raise ValueError("Fire start date column (DATE_DEBUT) not found. Available columns: " + str(list(fire_row.columns)))

print(f"Fire start date: {fire_start}")

# Define time range: 2 months before to 2 months after
time_start = fire_start - pd.DateOffset(months=2)
time_end = fire_start + pd.DateOffset(months=2)

print(f"Time range for data search: {time_start.strftime('%Y-%m-%d')} to {time_end.strftime('%Y-%m-%d')}")


In [ ]:
# Search and download HLS Sentinel-2 VI products using earthaccess
print("Authenticating with NASA Earthdata...")
# Note: You need to have your NASA Earthdata credentials set up
# You can set them via earthaccess.login() or have them in your .netrc file

try:
    earthaccess.login()
    print("Successfully authenticated with NASA Earthdata")
except Exception as e:
    print(f"Authentication failed: {e}")
    print("Please ensure you have NASA Earthdata credentials configured.")
    print("You can set them up by running: earthaccess.login() and following the prompts.")

# Convert datetime objects to strings for earthaccess
time_start_str = time_start.strftime('%Y-%m-%d')
time_end_str = time_end.strftime('%Y-%m-%d')

print(f"Searching for HLSS30_VI products from {time_start_str} to {time_end_str}")
print(f"Within bounding box: {bbox}")

# Search for HLSS30_VI (HLS Sentinel-2 Vegetation Indices) products
# Note: earthaccess uses 'bounding_box' parameter instead of 'bbox'
try:
    search_results = earthaccess.search_data(
        short_name='HLSS30_VI',  # HLS Sentinel-2 Vegetation Indices
        temporal=(time_start_str, time_end_str),
        bounding_box=bbox,  # Corrected parameter name for earthaccess
        count=200  # Limit results to avoid too many downloads
    )
except Exception as e:
    print(f"Error with bounding_box parameter: {e}")
    print("Trying alternative approach...")
    # Try without spatial filtering first to see if we get any results
    search_results = earthaccess.search_data(
        short_name='HLSS30_VI',
        temporal=(time_start_str, time_end_str),
        count=200
    )
    print(f"Found {len(search_results)} products without spatial filter")
    # If we have results, we'll need to filter them spatially later
    # For now, we'll proceed with what we have and handle filtering in processing

print(f"Found {len(search_results)} matching products.")

if len(search_results) == 0:
    print("No products found. Trying broader search or check date range.")
    # Try expanding the search slightly
    search_results = earthaccess.search_data(
        short_name='HLSS30_VI',
        temporal=(time_start_str, time_end_str),
        count=500
    )
    print(f"After broadening search: {len(search_results)} products found.")

In [ ]:
# Download the products
print(f"Downloading {len(search_results)} products to {OUTPUT_DIR}...")
downloaded_files = earthaccess.download(search_results, OUTPUT_DIR)

print(f"Successfully downloaded {len(downloaded_files)} files.")
print(f"Files saved in: {OUTPUT_DIR}")

# List downloaded files
hls_files = glob.glob(os.path.join(OUTPUT_DIR, "*.tif")) + glob.glob(os.path.join(OUTPUT_DIR, "*.TIF"))
print(f"Found {len(hls_files)} TIFF files for processing.")

if len(hls_files) == 0:
    print("No TIFF files found. Checking for other formats...")
    hls_files = glob.glob(os.path.join(OUTPUT_DIR, "*.hdf")) + glob.glob(os.path.join(OUTPUT_DIR, "*.HDF"))
    print(f"Found {len(hls_files)} HDF files.")

In [ ]:
# If the data were already downloaded, we can skip the download step and
#  just process the files in the output directory
# Initialize hls_files if not already defined (e.g., if running this cell separately)
if 'hls_files' not in globals():
    hls_files = []
if len(hls_files) == 0:
    hls_files = glob.glob(os.path.join(OUTPUT_DIR, "*.tif")) + glob.glob(os.path.join(OUTPUT_DIR, "*.TIF"))
    print(f"Re-checking for TIFF files: Found {len(hls_files)} files.")

In [ ]:
# Process HLS files and extract NDVI
print("Processing HLS files to extract NDVI...")

# Function to extract date from filename
def extract_date_from_filename(filename):
    # HLS filename format: HLS-VI.S30.TXXXXXXX.YYYYDDDThhmmss.*.tif
    date_match = re.search(r'\.(\d{4})(\d{3})T\d{6}\.', filename)
    if date_match:
        year = int(date_match.group(1))
        day_of_year = int(date_match.group(2))
        # Convert to datetime
        date = datetime(year, 1, 1) + timedelta(days=day_of_year-1)
        return date
    return None

# Filter files to only those containing NDVI in the filename (case-insensitive)
# More specific: look for .NDVI. pattern to avoid matching NDWI, etc.
ndvi_hls_files = sorted([f for f in hls_files if '.ndvi.' in os.path.basename(f).lower()])
print(f"Filtered to {len(ndvi_hls_files)} files containing '.NDVI.' in filename out of {len(hls_files)} total files.")

# Process each file
ndvi_data = []
valid_dates = []
ndvi_extents = []  # Store the geographic extent for each clipped NDVI array
fire_coverage_fractions = []  # Store fraction of fire boundary covered
valid_pixel_fractions = []  # Store fraction of valid pixels in each image

# Get the fire geometry in EPSG:4326 for clipping (using EXPANDED bounds for better overlap)
# This matches what we used for the earthaccess search
if fire_gdf.crs != "EPSG:4326":
    # Create a temporary GeoSeries with the fire geometry to reproject it
    fire_geom_series = GeoSeries([fire_row.geometry.iloc[0]], crs=fire_gdf.crs)
    fire_geom_wgs84 = fire_geom_series.to_crs("EPSG:4326").iloc[0]
    minx_orig, miny_orig, maxx_orig, maxy_orig = fire_geom_wgs84.bounds
else:
    minx_orig, miny_orig, maxx_orig, maxy_orig = fire_row.geometry.iloc[0].bounds

# Calculate expanded bounds (2x width and height) - SAME AS USED IN EARTHACCESS SEARCH
center_x = (minx_orig + maxx_orig) / 2
center_y = (miny_orig + maxy_orig) / 2
width = maxx_orig - minx_orig
height = maxy_orig - miny_orig

# Expand to twice the width and height
new_width = width * 2
new_height = height * 2
new_minx = center_x - new_width / 2
new_maxx = center_x + new_width / 2
new_miny = center_y - new_height / 2
new_maxy = center_y + new_height / 2

# Use expanded bounds for clipping to ensure overlap with searched data
minx_clip, miny_clip, maxx_clip, maxy_clip = new_minx, new_miny, new_maxx, new_maxy

print(f"Filtering NDVI data to EXPANDED fire bounds: ({minx_clip:.6f}, {miny_clip:.6f}) to ({maxx_clip:.6f}, {maxy_clip:.6f})")
print(f"  Original fire bounds: ({minx_orig:.6f}, {miny_orig:.6f}) to ({maxx_orig:.6f}, {maxy_orig:.6f})")

# Also get the original fire geometry in EPSG:4326 for overlay
if fire_gdf.crs != "EPSG:4326":
    fire_geometry_wgs84 = fire_geom_wgs84  # Already calculated above
else:
    fire_geometry_wgs84 = fire_row.geometry.iloc[0]

for file_path in ndvi_hls_files:
    try:
        # Extract date from filename
        filename = os.path.basename(file_path)
        date = extract_date_from_filename(filename)
        if date is None:
            print(f"Could not extract date from {filename}, skipping")
            continue

        # Open the file with rioxarray
        print(f"Processing {filename} for date {date}")
        ds = rxr.open_rasterio(file_path, masked=True)
        
        # Extract cloud coverage from metadata
        cloud_coverage = ds.attrs.get('cloud_coverage')
        if cloud_coverage is None:
            tags = ds.tags()
            cloud_coverage = tags.get('cloud_coverage')
        if cloud_coverage is not None:
            print(f"  Cloud coverage: {cloud_coverage}%")
        else:
            print(f"  Cloud coverage: Not found in metadata")
        
        # Debug: Check CRS and bounds of the raster
        print(f"  Raster CRS: {ds.rio.crs}")
        if hasattr(ds, 'rio') and ds.rio.crs:
            raster_bounds = ds.rio.bounds()
            print(f"  Raster bounds: ({raster_bounds[0]:.2f}, {raster_bounds[1]:.2f}) to ({raster_bounds[2]:.2f}, {raster_bounds[3]:.2f})")
            print(f"  Fire bounds for clipping (EPSG:4326): ({minx_clip:.6f}, {miny_clip:.6f}) to ({maxx_clip:.6f}, {maxy_clip:.6f})")
            
            # Check if there's overlap by converting fire bounds to raster CRS if needed
            if raster_bounds and ds.rio.crs:
                try:
                    # Create a GeoSeries for the fire bounds and reproject to raster CRS
                    fire_box = box(minx_clip, miny_clip, maxx_clip, maxy_clip)
                    fire_gs = GeoSeries([fire_box], crs="EPSG:4326")
                    
                    # Reproject fire bounds to match raster CRS
                    if str(fire_gs.crs) != str(ds.rio.crs):
                        fire_gs_reprojected = fire_gs.to_crs(ds.rio.crs)
                        fire_bounds_reprojected = fire_gs_reprojected.iloc[0].bounds
                        print(f"  Fire bounds reprojected to raster CRS: ({fire_bounds_reprojected[0]:.2f}, {fire_bounds_reprojected[1]:.2f}) to ({fire_bounds_reprojected[2]:.2f}, {fire_bounds_reprojected[3]:.2f})")
                        
                        # Check for overlap
                        raster_minx, raster_miny, raster_maxx, raster_maxy = raster_bounds
                        fire_minx, fire_miny, fire_maxx, fire_maxy = fire_bounds_reprojected
                        
                        overlap_x = not (fire_maxx < raster_minx or fire_minx > raster_maxx)
                        overlap_y = not (fire_maxy < raster_miny or fire_miny > raster_maxy)
                        has_overlap = overlap_x and overlap_y
                        print(f"  Bounds overlap check: {has_overlap} (x: {overlap_x}, y: {overlap_y})")
                        
                        if not has_overlap:
                            print(f"  WARNING: No overlap between raster and fire bounds!")
                            print(f"    Raster X range: [{raster_minx:.2f}, {raster_maxx:.2f}]")
                            print(f"    Fire X range: [{fire_minx:.2f}, {fire_maxx:.2f}]")
                            print(f"    Raster Y range: [{raster_miny:.2f}, {raster_maxy:.2f}]")
                            print(f"    Fire Y range: [{fire_miny:.2f}, {fire_miny:.2f}]")
                        else:
                            # Use the reprojected fire bounds for clipping
                            clip_bounds = fire_bounds_reprojected
                    else:
                        clip_bounds = (minx_clip, miny_clip, maxx_clip, maxy_clip)
                        
                except Exception as e:
                    print(f"  Error checking bounds overlap: {e}")
                    # Fallback to original approach
                    clip_bounds = (minx_clip, miny_clip, maxx_clip, maxy_clip)
            else:
                clip_bounds = (minx_clip, miny_clip, maxx_clip, maxy_clip)
        else:
            clip_bounds = (minx_clip, miny_clip, maxx_clip, maxy_clip)

        # Since we filtered for NDVI files, the dataset contains NDVI data directly
        # Handle different dataset structures
        if hasattr(ds, 'data'):
            # xarray Dataset with data variable
            ndvi = ds
        elif hasattr(ds, 'variables') and len(ds.variables) > 0:
            # Check if there's a data variable
            data_vars = [v for v in ds.variables if v not in ['x', 'y', 'band', 'lat', 'lon']]
            if data_vars:
                ndvi = ds[data_vars[0]]
            else:
                ndvi = ds
        else:
            # Assume the dataset itself is the data
            ndvi = ds

        # Clip to fire bounding box (using appropriate bounds for overlap)
        print(f"  Attempting to clip to fire bounds...")
        ndvi_clipped = ndvi.rio.clip_box(*clip_bounds)
        
        # Check if clipping resulted in empty data
        if ndvi_clipped.size == 0 or ndvi_clipped.values.size == 0:
            print(f"  WARNING: Clipping resulted in empty data!")
            continue

        # Calculate fraction of fire boundary covered by clipped data
        try:
            # Get the clipped data bounds in its native CRS
            clipped_bounds = ndvi_clipped.rio.bounds()
            # Create a box from the clipped bounds
            clipped_geom = box(*clipped_bounds)
            
            # Get fire geometry in the same CRS as the raster
            if fire_gdf.crs != ds.rio.crs:
                # Reproject fire geometry to match raster CRS
                fire_gs = GeoSeries([fire_row.geometry.iloc[0]], crs=fire_gdf.crs)
                fire_geom_raster_crs = fire_gs.to_crs(ds.rio.crs).iloc[0]
            else:
                fire_geom_raster_crs = fire_row.geometry.iloc[0]
            
            # Calculate intersection
            intersection = fire_geom_raster_crs.intersection(clipped_geom)
            
            # Calculate areas
            fire_area = fire_geom_raster_crs.area
            intersection_area = intersection.area
            
            # Avoid division by zero
            if fire_area > 0:
                coverage_fraction = (intersection_area / fire_area) * 100  # as percentage
                print(f"  Fire area: {fire_area:.2f}, Intersection area: {intersection_area:.2f}")
                print(f"  Fire boundary coverage: {coverage_fraction:.1f}%")

                # Apply user-specified threshold for fire coverage
                if coverage_fraction < FIRE_COVERAGE_THRESHOLD:
                    print(f"  Skipping image due to insufficient fire coverage ({coverage_fraction:.1f}% < {FIRE_COVERAGE_THRESHOLD}%)")
                    continue  # Skip to next file

                fire_coverage_fractions.append(coverage_fraction)

                # Calculate fraction of valid pixels in the clipped data
                # Count total pixels and valid (non-masked/non-NaN) pixels
                total_pixels = ndvi_clipped.size
                valid_pixels = ndvi_clipped.count().values
                valid_pixel_fraction = valid_pixels / total_pixels if total_pixels > 0 else 0.0
                valid_pixel_percentage = valid_pixel_fraction * 100  # Convert to percentage
                valid_pixel_fractions.append(valid_pixel_percentage)
                print(f"  Valid pixel fraction: {valid_pixel_percentage:.1f}% ({valid_pixels}/{total_pixels} pixels)")

                # Apply user-specified threshold for valid pixels
                if valid_pixel_percentage < VALID_PIXEL_THRESHOLD:
                    print(f"  Skipping image due to insufficient valid pixels ({valid_pixel_percentage:.1f}% < {VALID_PIXEL_THRESHOLD}%)")
                    # Remove the fire coverage fraction we just added since we're skipping this image
                    fire_coverage_fractions.pop()
                    continue  # Skip to next file

            else:
                print(f"  WARNING: Fire area is zero or invalid")
                fire_coverage_fractions.append(0.0)
                valid_pixel_fractions.append(0.0)  # Append default value
                
        except Exception as e:
            print(f"  WARNING: Could not calculate fire coverage: {e}")
            fire_coverage_fractions.append(0.0)
            valid_pixel_fractions.append(0.0)  # Append default value

        # Convert to proper NDVI scale if needed (HLS data is often scaled)
        # Check if data appears to be scaled (values outside typical NDVI range -1 to 1)
        data_min = float(ndvi_clipped.min().values)
        data_max = float(ndvi_clipped.max().values)
        
        # If data appears to be scaled (commonly by 10000 for NDVI in satellite data)
        if abs(data_min) > 100 or abs(data_max) > 100:
            print(f"  Data appears scaled (range: {data_min:.0f} to {data_max:.0f}), converting to NDVI scale")
            # Apply scaling factor (typically 10000 for HLS NDVI)
            ndvi_clipped = ndvi_clipped / 10000.0
            # Update min/max after scaling
            data_min = float(ndvi_clipped.min().values)
            data_max = float(ndvi_clipped.max().values)
            print(f"  After scaling: range {data_min:.4f} to {data_max:.4f}")

        # Get the geographic extent of the clipped data for proper plotting
        # This will be in the CRS of the raster data
        try:
            # Get the bounding box of the clipped data in its native CRS
            clipped_bounds = ndvi_clipped.rio.bounds()
            # Convert to EPSG:4326 for consistent plotting
            if hasattr(ndvi_clipped, 'rio') and ndvi_clipped.rio.crs:
                from shapely.geometry import box
                bbox_geom = box(*clipped_bounds)
                bbox_gs = GeoSeries([bbox_geom], crs=ndvi_clipped.rio.crs)
                bbox_wgs84 = bbox_gs.to_crs("EPSG:4326").iloc[0]
                extent = (bbox_wgs84.bounds[0], bbox_wgs84.bounds[2], bbox_wgs84.bounds[1], bbox_wgs84.bounds[3])  # (west, east, south, north)
                print(f"  Clipped extent (WGS84): {extent}")
            else:
                extent = None
                print(f"  WARNING: Could not determine extent for clipping")
        except Exception as e:
            print(f"  WARNING: Error getting extent: {e}")
            extent = None

        # Store the data
        ndvi_data.append(ndvi_clipped.values)
        valid_dates.append(date)
        ndvi_extents.append(extent)

        print(f"  Extracted NDVI shape: {ndvi_clipped.shape}")
        print(f"  Data range: {data_min:.4f} to {data_max:.4f}")
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        continue

print(f"Successfully processed {len(ndvi_data)} files with valid NDVI data.")
if fire_coverage_fractions:
    avg_coverage = sum(fire_coverage_fractions) / len(fire_coverage_fractions)
    print(f"Average fire boundary coverage across all frames: {avg_coverage:.1f}%")
if valid_pixel_fractions:
    avg_valid_pixels = sum(valid_pixel_fractions) / len(valid_pixel_fractions)
    print(f"Average valid pixel fraction across all frames: {avg_valid_pixels:.1f}%")

In [ ]:
# Create timelapse animation
print("Creating timelapse animation...")

if len(ndvi_data) == 0:
    print("No NDVI data to create animation.")
else:
    # Sort data by date
    sorted_indices = np.argsort(valid_dates)
    ndvi_data_sorted = [ndvi_data[i] for i in sorted_indices]
    valid_dates_sorted = [valid_dates[i] for i in sorted_indices]
    ndvi_extents_sorted = [ndvi_extents[i] for i in sorted_indices] if len(ndvi_extents) > 0 else []
    
    print(f"Creating animation with {len(ndvi_data_sorted)} frames from {valid_dates_sorted[0]} to {valid_dates_sorted[-1]}")

    # Set up the figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Define NDVI colormap (brown to yellow to green) - FIXED for proper NDVI mapping
    # NDVI range: -1 (brown/bare soil) to 0 (yellow) to +1 (green/healthy vegetation)
    ndvi_cmap = colors.LinearSegmentedColormap.from_list(
        'ndvi_colormap',
        [(0.4, 0.2, 0.1),   # Dark brown for NDVI = -1
         (0.8, 0.6, 0.2),   # Brown-yellow for NDVI = -0.5
         (0.9, 0.9, 0.4),   # Pale yellow for NDVI = 0
         (0.6, 0.8, 0.3),   # Light green for NDVI = 0.5
         (0.1, 0.6, 0.1)],  # Dark green for NDVI = +1
        N=256
    )
    
    # Initialize the plot - squeeze the first frame to remove singleton dimensions
    first_frame = np.squeeze(ndvi_data_sorted[0])
    
    # Determine extent for the first frame
    if len(ndvi_extents_sorted) > 0 and ndvi_extents_sorted[0] is not None:
        extent = ndvi_extents_sorted[0]
        im = ax.imshow(first_frame, cmap=ndvi_cmap, vmin=-1, vmax=1, extent=extent)
        print(f"Using geographic extent: {extent}")
    else:
        im = ax.imshow(first_frame, cmap=ndvi_cmap, vmin=-1, vmax=1)
        print("Using pixel coordinates (no geographic extent available)")
    
    ax.set_title(f'NDVI Timelapse: {valid_dates_sorted[0].strftime("%Y-%m-%d")}')
    
    # Add colorbar with proper labeling and extension indicators
    cbar = plt.colorbar(im, ax=ax, label='NDVI', extend='both')
    cbar.set_ticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    cbar.set_ticklabels(['-1.0', '-0.5', '0.0', '0.5', '+1.0'])
    
    # Add fire geometry overlay if available
    if 'fire_geometry_wgs84' in globals() and fire_geometry_wgs84 is not None:
        try:
            # Convert fire geometry to GeoSeries for plotting
            fire_gs = GeoSeries([fire_geometry_wgs84], crs="EPSG:4326")
            # Plot the fire boundary
            fire_gs.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2, label='Fire Boundary')
            ax.legend()
            print("Added fire geometry overlay")
        except Exception as e:
            print(f"Warning: Could not overlay fire geometry: {e}")
    
    # Set labels for geographic coordinates
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    # Animation function
    def animate(frame):
        # Squeeze the data to remove singleton dimensions
        data_squeezed = np.squeeze(ndvi_data_sorted[frame])
        im.set_array(data_squeezed)
        
        # Update extent if available
        if len(ndvi_extents_sorted) > frame and ndvi_extents_sorted[frame] is not None:
            im.set_extent(ndvi_extents_sorted[frame])
        
        ax.set_title(f'NDVI Timelapse: {valid_dates_sorted[frame].strftime("%Y-%m-%d")}')
        return [im]

    # Create animation
    anim = animation.FuncAnimation(fig, animate, frames=len(ndvi_data_sorted),
                                 interval=500, blit=True, repeat=True)

    # Save animation
    print(f"Saving animation to {ANIMATION_OUTPUT}...")
    anim.save(ANIMATION_OUTPUT, writer='pillow', fps=2)
    
    plt.close(fig)
    print("Animation saved successfully!")